# Compass Dashboard

In [2]:
import pandas as pd
import pdfplumber
import sys
import os
sys.path.append(os.path.abspath(".."))


## Preprocessing

Like many mid-sized companies, my groundhandling agency at Vancouver airport loves to distribute employee rosters in the form of pdf documents with schedule laid out in unloveable tables that are neither machine readable nor copy-paste-able. To first extract shift information from these tables, I used the pdfplumber library, which contains a range of functions for dealing with pdf documents.

In [3]:
with pdfplumber.open("../data/Monthly Schedule _ MAY.pdf") as pdf:
    page = pdf.pages[0]
    table = page.extract_table()
pd.DataFrame(table).head()

,0,1,2,3,4,5,6,7
0,MAY 2026,Mon\n04-May,Tue\n05-May,Wed\n06-May,Thu\n07-May,Fri\n08-May,Sat\n09-May,Sun\n10-May
1,Ratna\nSalim,,,1715-2115 GT1,,1715-2115 GT1,,
2,Alex\nYe,1715-2115 GT1,,,1330-2100\nARR+DEP,1715-2115 GT1,1715-2115 GT1,1715-2115 GT1
3,Ann\nKim,,1630-2030 TKT,1630-2030 TKT,1630-2030 TKT,1630-2030 TKT,,1630-2030 TKT
4,Saeko\nSumi,1415-2115 FC,1415-2115 FC,1415-2115 FC,,1415-2115 FC,1415-2115 FC,


So in fact with this specific document it is quite easy to use extract_table and turn it directly into a data frame. However, with PDF documents this is not always the case, and a more robust solution that is good to practice is to extract words with their positions and reconstruct the table using those relative postions. My function below uses pdfplumber's extract words to retrieve all text instances from the table along with their x and y coordinates, and then reconstructs the information regarding my shifts into a dictionary for later use

In [4]:
from src.parser.orchestrator import parse_pdf

In [6]:
from src.parser.pdf_reader import extract_words_from_pdf
from src.parser.extractors import get_date_columns, get_employee_shifts, find_employee_rows, assign_dates, match_shift_types

In [7]:
def parse_pdf(pdf_path):
    all_results = []

    for words in extract_words_from_pdf(pdf_path):

        date_columns = get_date_columns(words)
        employee_rows = find_employee_rows(words)

        shifts = get_employee_shifts(
            words, employee_rows['shiftTimeRow']
            )
        shifts_w_type = match_shift_types(
            shifts, words, employee_rows['shiftTimeRow']
            )
        results = assign_dates(shifts_w_type, date_columns)

        all_results.extend(results)

    return all_results

In [8]:

data = parse_pdf('../data/Monthly Schedule _ MAY.pdf')
data[:5]

[{'text': '1715-2115', 'x0': 149.66, 'x1': 194.46008, 'top': 566.27872, 'doctop': 566.27872, 'bottom': 576.2387200000001, 'upright': True, 'height': 9.960000000000036, 'width': 44.80008000000001, 'direction': 'ltr'}, {'text': '1715-2115', 'x0': 221.81, 'x1': 266.61008000000004, 'top': 566.27872, 'doctop': 566.27872, 'bottom': 576.2387200000001, 'upright': True, 'height': 9.960000000000036, 'width': 44.80008000000004, 'direction': 'ltr'}]
SHIFT TYPE ROW TARGET: 560.34328
Candidate near x=149.66: text=1630-2030, top=134.22871999999995
Candidate near x=149.66: text=1415-2115, top=158.22871999999995
Candidate near x=149.66: text=1315-2115, top=182.22871999999995
Candidate near x=149.66: text=1330-2100, top=302.25872000000004
Candidate near x=149.66: text=1715-2115, top=398.25872000000004
Candidate near x=149.66: text=1715-2115, top=446.27872
Candidate near x=149.66: text=1315-2115, top=470.27872
Candidate near x=149.66: text=1715-2115, top=566.27872
Candidate near x=149.66: text=1715-2115,

[]

In [ ]:
df = pd.DataFrame(data)
df.head()

This is all the information that I need parsed correctly from the original document, but its current format leaves a bit to be desired. The `clean_dataframe` function below splits the time column into separate start and end times, as well as standardizing the spacing and capitalisation of all values and column names as a way to future-proof the code. It also performs a quick validation of the data and will raise an error if there are any missing dates or time values

In [ ]:
from src.processing.cleaning import clean_dataframe

df = clean_dataframe(df)

df.head()

Now the data is in a much more useable form, but the dates are still just strings. An obvious improvement would be to transform them into datetime objects, which is what the `add_datetime_columns` function does. Note that it takes a start year for the schedule and will automatically increment the year in case the date range spans January 1st.

In [ ]:
from src.processing.datetime_utils import add_datetime_columns, parse_times

df = add_datetime_columns(df)
df = parse_times(df)

df.head()

And from there I can add a few extra features that will be useful when calculating transit costs, using only the datetimes in the first column.

In [ ]:
from src.processing.features import add_features

df = add_features(df)

df.head()

Wrapping this all together into a pipeline I can create this dataframe with a single function call.

In [ ]:
from src.pipeline import run_pipeline

df = run_pipeline('../data/Monthly Schedule _ MAY.pdf', start_year=2026)

df.head()


In [ ]:
from src.logic.trips import ShiftsToTrips

In [ ]:
trips = ShiftsToTrips(df, 90)
trips.head()

In [ ]:
trips.shape

In [ ]:
from src.logic.trips import GetOneZoneTrips, count_trip_types

In [ ]:
GetOneZoneTrips(trips).head()

In [ ]:
counts = count_trip_types(trips)
pd.DataFrame.from_dict(counts, orient='index', columns=['Trips'])

In [ ]:
from src.config.fare_prices import FARES

In [ ]:
FARES.one_zone

In [ ]:
from src.logic.fares import cost_one_zone_pass

In [ ]:
cost_one_zone_pass(counts,FARES)

In [ ]:
from src.logic.costs import total_costs

In [ ]:
total_costs(df, FARES, 60)['costs']